# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [18]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [19]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC1:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc1,
)
print(
    "Parâmetro de filtragem median binding percentile TCC1:.........",
    parametros.parametro_de_filtragem_mbp_tcc1,
)
print(
    "Percentual de match mínimo TCC1:...............................",
    parametros.percent_match_minimo_tcc1,
)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 1
Percentual de match mínimo TCC1:............................... 95.0


In [20]:
import pandas as pd

In [21]:
df = pd.read_csv('./T CELL/DENV 3 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VSGKLIHEW,303,311,9,HLA-B*57:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.997373,0.01
1,1,VSGKLIHEW,303,311,9,HLA-B*58:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.996354,0.01
2,1,TTRMENLLW,60,68,9,HLA-B*57:01,60,0.01,TTRMENLLW,TTRMENLLW,0.990271,0.01
3,1,CTWPKSHTLW,223,232,10,HLA-B*57:01,567,0.01,CTWPKSHTW,CTWPKSHTLW,0.989558,0.01
4,1,EVHTWTEQY,24,32,9,HLA-A*26:01,24,0.01,EVHTWTEQY,EVHTWTEQY,0.988494,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,PISEKEENMVKS,337,348,12,HLA-A*24:02,1366,100.00,PISEKMVKS,PISEKEENMVKS,0.000000,100.00
36986,1,PISEKEENMVKS,337,348,12,HLA-A*32:01,1366,100.00,PISENMVKS,PISEKEENMVKS,0.000000,100.00
36987,1,PISEKEENMVKS,337,348,12,HLA-B*35:01,1366,100.00,SEKEENVKS,SEKEENMVKS,0.000000,100.00
36988,1,PISEKEENMVKS,337,348,12,HLA-B*53:01,1366,100.00,PISEKEVKS,PISEKEENMVKS,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [22]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc1

In [23]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  1


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VSGKLIHEW,303,311,9,HLA-B*57:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.997373,0.01
1,1,VSGKLIHEW,303,311,9,HLA-B*58:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.996354,0.01
2,1,TTRMENLLW,60,68,9,HLA-B*57:01,60,0.01,TTRMENLLW,TTRMENLLW,0.990271,0.01
3,1,CTWPKSHTLW,223,232,10,HLA-B*57:01,567,0.01,CTWPKSHTW,CTWPKSHTLW,0.989558,0.01
4,1,EVHTWTEQY,24,32,9,HLA-A*26:01,24,0.01,EVHTWTEQY,EVHTWTEQY,0.988494,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
604,1,VKTCTWPKSHTL,220,231,12,HLA-B*57:01,1249,0.99,VTWPKSHTL,VKTCTWPKSHTL,0.277652,0.99
605,1,RAVHADMGYWI,192,202,11,HLA-B*58:01,879,0.99,RAVHADMYW,RAVHADMGYW,0.146511,0.99
606,1,DSPKRLATA,37,45,9,HLA-B*08:01,37,0.99,DSPKRLATA,DSPKRLATA,0.117196,0.99
607,1,VTAETQNSSF,124,133,10,HLA-A*01:01,468,0.99,VTETQNSSF,VTAETQNSSF,0.093092,0.99


## Agrupando por pepitideos e agregando colunas pertinentes

In [24]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADSPKRLATAI,36,46,1,0.520,HLA-B*07:02
1,AETQNSSFI,126,134,3,0.250,"HLA-B*40:01, HLA-B*44:02, HLA-B*44:03"
2,AETQNSSFII,126,135,3,0.730,"HLA-B*40:01, HLA-B*44:02, HLA-B*44:03"
3,AGPISQHNY,248,256,1,0.450,HLA-A*30:02
4,AGPISQHNYR,248,257,1,0.870,HLA-A*31:01
...,...,...,...,...,...,...
232,YGFGVFTTNIW,158,168,3,0.380,"HLA-B*53:01, HLA-B*57:01, HLA-B*58:01"
233,YILWENNIKL,77,86,3,0.120,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06"
234,YILWENNIKLT,77,87,1,0.570,HLA-A*02:01
235,YMGEDGCWY,323,331,2,0.675,"HLA-A*01:01, HLA-B*15:01"


## Filtragem por qte_de_alelos

In [25]:
# mbp_minimo = 1
qte_alelos_minimo = parametros.filtarar_por_qte_de_alelos_tcc1

In [26]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_alelos_minimo
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,CTWPKSHTL,223,231,10,0.340,"HLA-A*02:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2..."
1,HTWTEQYKF,26,34,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
2,QPMELKYSW,107,115,10,0.505,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3..."


## Sorting por median_biding_percentile

In [27]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,HTWTEQYKF,26,34,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,CTWPKSHTL,223,231,10,0.340,"HLA-A*02:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2..."
2,QPMELKYSW,107,115,10,0.505,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [28]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_1.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0    HTWTEQYKF
1    CTWPKSHTL
2    QPMELKYSW
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [29]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [30]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_1.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,HTWTEQYKF,9,99.06% (632/638),88.89%,100.00%,NaN
1,2,NP 2,CTWPKSHTL,9,98.75% (630/638),88.89%,100.00%,NaN
2,3,NP 3,QPMELKYSW,9,99.22% (633/638),88.89%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [31]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "median_binding_percentile"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos
0,NP 1,HTWTEQYKF,9,99.06% (632/638),88.89%,100.00%,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,NP 2,CTWPKSHTL,9,98.75% (630/638),88.89%,100.00%,10,0.340,"HLA-A*02:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2..."
2,NP 3,QPMELKYSW,9,99.22% (633/638),88.89%,100.00%,10,0.505,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [32]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,NP 1,HTWTEQYKF,9,99.06% (632/638),88.89%,100.00%,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.06
1,NP 2,CTWPKSHTL,9,98.75% (630/638),88.89%,100.00%,10,0.340,"HLA-A*02:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2...",98.75
2,NP 3,QPMELKYSW,9,99.22% (633/638),88.89%,100.00%,10,0.505,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3...",99.22


### Sort e filtragem por percent_match e presença em alelos

In [33]:
minimum_percent_match = parametros.percent_match_minimo_tcc1

In [34]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= minimum_percent_match]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,NP 3,QPMELKYSW,9,99.22% (633/638),88.89%,100.00%,10,0.505,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3...",99.22
1,NP 1,HTWTEQYKF,9,99.06% (632/638),88.89%,100.00%,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.06
2,NP 2,CTWPKSHTL,9,98.75% (630/638),88.89%,100.00%,10,0.340,"HLA-A*02:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2...",98.75
